#### Preprocessing code

Code to convert MATLAB processing into Python

Created for dlx56_mPFC_1p_SohalLab repo
based off code in ruleshifting-inscopix private repo

#### Import packages


In [5]:
from __future__ import annotations

import os
from pathlib import Path
from datetime import datetime
from typing import List, Any, Dict
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed


In [6]:
#import local yaml for env preset variables
from analysis_config_loader import load_analysis_config  # new import
config_path = "analysis_config.yaml"  # adjust as needed
analysis_config = load_analysis_config(config_path)

TypeError: AnalysisConfig.__init__() missing 5 required positional arguments: 'use_dff_not_spikes', 'zscore_dff', 'zscore_dff_to_baseline', 'spike_decay_frac_of_peak_cutoff', and 'end_event_at_drop_frac_of_diff'

In [ ]:
analysis_config

#### Define dataset methods

In [ ]:
# ---------- Project specific hooks (unchanged stubs) ----------
class DatasetObject:
    def __init__(self, name: str, raster: np.ndarray):
        self.name = name
        self.raster = raster

class DatasetFileMethods:
    @staticmethod
    def get_root_dir(server_use: bool) -> Path:
        return Path("/sohal1/cjcruz")

    @staticmethod
    def make_storage_folder_name(root_dir: Path, name: str) -> Path:
        safe = "".join(c if c.isalnum() or c in " _-." else "_" for c in name)
        return root_dir / "analysis_outputs" / safe

    @staticmethod
    def get_dataset_folder_dir(server_use: bool, data_type_used: str) -> Path:
        return Path("/sohal1/cjcruz/Dlx mice inscopix/dataset_object_storage_main")

    @staticmethod
    def get_list_datasets_in_folder(folder: Path) -> List[str]:
        return sorted([p.name for p in folder.iterdir() if p.is_dir()])

    @staticmethod
    def get_dataset_object_i(dataset_path: Path, data_type_used: str) -> DatasetObject:
        raise NotImplementedError

def return_binary_activity_vec_sig_active_in_task_stage(dataset_object: DatasetObject) -> pd.DataFrame:
    raise NotImplementedError

In [ ]:
# set up locations for input/output
server_use = True
data_type_used = "neurons"
run_activity_enrichment = True
export_mean_active = True
use_WT_CLNZ_folder = False

#get where to load data 
dfm = DatasetFileMethods()
root_dir = dfm.get_root_dir(server_use)
os.chdir(root_dir)

#make analysis folder 
hour_str = datetime.now().strftime("%H")
analysis_name = f" sig enrichment vectors by task stage{analysis_config.num_shuffles}_shuffles_hour_{hour_str}"
storage_folder_name = dfm.make_storage_folder_name(root_dir, f"{data_type_used}_{analysis_name}")
storage_folder_name.mkdir(parents=True, exist_ok=True)


In [ ]:

if use_WT_CLNZ_folder:
    dataset_folders = Path("/sohal1/cjcruz/Dlx mice inscopix/dataset_object_storage_WT_CLNZ")
else:
    dataset_folders = dfm.get_dataset_folder_dir(server_use, data_type_used)

phase_type = analysis_config.phase_division_types[1]  # 'complex'
content_names = dfm.get_list_datasets_in_folder(dataset_folders)


#### Run activity enrichment detection

In [ ]:
if run_activity_enrichment:
    def _job(name: str) -> pd.DataFrame:
        ds = dfm.get_dataset_object_i(dataset_folders / name, data_type_used)
        return return_binary_activity_vec_sig_active_in_task_stage(ds)

    tables: List[pd.DataFrame] = [None] * len(content_names)
    with ProcessPoolExecutor() as ex:
        futs = {ex.submit(_job, nm): i for i, nm in enumerate(content_names)}
        for fut in as_completed(futs):
            tables[futs[fut]] = fut.result()

    needed_cols = analysis_config.get_num_phase_pairs(phase_type) + 2
    fixed = []
    for df in tables:
        if df.shape[1] < needed_cols and "baseline" not in df.columns:
            df = df.copy()
            df["baseline"] = 0
        fixed.append(df)

    combined = pd.concat(fixed, axis=0, ignore_index=True)
    metadata_vec = [
        str(datetime.now()),
        analysis_config.num_shuffles,
        analysis_config.percentile,
        analysis_config.drop_low_value_peak_events,
        analysis_config.cutoff_filter,
        analysis_config.peak_event_cutoff_percentile,
    ]
    meta_df = pd.DataFrame([metadata_vec] * len(combined))
    combined = pd.concat([combined, meta_df], axis=1)

    dataset_cohort = "WT_CLNZ_" if use_WT_CLNZ_folder else "main_datasets_"
    out_name = f"{dataset_cohort}{phase_type}TACO- {data_type_used}_activity by task phase_{datetime.now().date()}.xlsx"
    combined.to_excel(out_name, index=False)

if export_mean_active:
    mean_tables: List[pd.DataFrame] = []
    for i, nm in enumerate(content_names, start=1):
        print(f"{i}/{len(content_names)} mean activity found")
        ds = dfm.get_dataset_object_i(dataset_folders / nm, data_type_used)
        curr_mean = ds.raster.mean(axis=1)
        names = np.full(curr_mean.shape[0], ds.name)
        mean_tables.append(pd.DataFrame({"curr_mean_active": curr_mean, "name_col": names}))

    activation_table = pd.concat(mean_tables, axis=0, ignore_index=True)
    out_name2 = f"{data_type_used}_mean dataset activity_{datetime.now().date()}.xlsx"
    activation_table.to_excel(out_name2, index=False)

print("Done creating all thresholded activity vectors for all datasets. Exported")
